#### Modeling

Setup and load data

In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import math
import statsmodels.api as sm  

In [4]:
# load data
X_train = pd.read_csv("../data/X_train_movies.csv")
X_test  = pd.read_csv("../data/X_test_movies.csv")

y_train = pd.read_csv("../data/y_train_movies.csv")
y_test  = pd.read_csv("../data/y_test_movies.csv")

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8204, 26) (6090, 26) (8204, 1) (6090, 1)


Define evaluation function

In [5]:
def score_regression(y_true, y_pred):
    """
    Return R^2, MAE, and RMSE for a regression model.
    """
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    
    return pd.Series({"R2": r2, "MAE": mae, "RMSE": rmse})

Fit 3 linear regression models on the training data:

• Model 1: Use only runtime_in_minutes

• Model 2: Use runtime_in_minutes and kid_friendly

• Model 3: Use runtime_in_minutes, kid_friendly and the dummy columns for the genres

Choose feature sets for the three models

In [6]:
# Model 1: runtime only
feat_model1 = ["runtime_in_minutes"]

# Model 2: runtime + kid_friendly
feat_model2 = ["runtime_in_minutes", "kid_friendly"]

# Model 3: runtime + kid_friendly + all genre dummies
exclude_from_genre = [
    "runtime_in_minutes",
    "kid_friendly",
    "kid_family",
    "days_to_stream",
    "holiday_blockbuster",
]

genre_cols = [c for c in X_train.columns if c not in exclude_from_genre]

feat_model3 = ["runtime_in_minutes", "kid_friendly"] + genre_cols

len(genre_cols), genre_cols[:10]  # just to sanity-check

(21,
 ['Animation',
  'Anime & Manga',
  'Art House & International',
  'Classics',
  'Comedy',
  'Cult Movies',
  'Documentary',
  'Drama',
  'Faith & Spirituality',
  'Gay & Lesbian'])

Fit the three linear models and score them

In [9]:
results = {}

# Model 1
m1 = LinearRegression()
m1.fit(X_train[feat_model1], y_train)
y_pred1 = m1.predict(X_test[feat_model1])
results["Model 1: runtime only"] = score_regression(y_test, y_pred1)

# Model 2 
m2 = LinearRegression()
m2.fit(X_train[feat_model2], y_train)
y_pred2 = m2.predict(X_test[feat_model2])
results["Model 2: runtime + kid_friendly"] = score_regression(y_test, y_pred2)

# Model 3 
m3 = LinearRegression()
m3.fit(X_train[feat_model3], y_train)
y_pred3 = m3.predict(X_test[feat_model3])
results["Model 3: runtime + kid_friendly + genres"] = score_regression(y_test, y_pred3)

# all metrics into one DataFrame
model_scores = (
    pd.DataFrame(results)
    .T  # models as rows
)

model_scores

,R2,MAE,RMSE
Model 1: runtime only,-0.023767,25.071866,28.711103
Model 2: runtime + kid_friendly,-0.025618,25.105363,28.737037
Model 3: runtime + kid_friendly + genres,0.030971,24.001836,27.933010


Model 3 (runtime_in_minutes + kid_friendly + genre dummies) performs the best: it has the highest R^2 (~0.03) and the lowest MAE and RMSE compared with Models 1 and 2. The improvement is small, but it’s the only model that beats the baseline a little.

In [10]:
# Statsmodels for p-values
X3_train_sm = sm.add_constant(X_train[feat_model3])
ols3 = sm.OLS(y_train, X3_train_sm).fit()
print(ols3.summary())

                            OLS Regression Results                            
Dep. Variable:          critic_rating   R-squared:                       0.130
Model:                            OLS   Adj. R-squared:                  0.127
Method:                 Least Squares   F-statistic:                     53.05
Date:                Tue, 02 Dec 2025   Prob (F-statistic):          3.96e-226
Time:                        01:05:33   Log-Likelihood:                -38434.
No. Observations:                8204   AIC:                         7.692e+04
Df Residuals:                    8180   BIC:                         7.708e+04
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                 

From the statsmodels results, the most useful predictors are the ones with very small p-values. Runtime_in_minutes and kid_friendly are both highly significant and positive, so longer films and G/PG movies tend to receive higher critic ratings. Among the genre dummies, Animation, Art House & International, Classics, Comedy, Documentary, Drama, Musical & Performing Arts, Special Interest, Sports & Fitness, Television, and Western all have small p-values and positive coefficients, meaning those genres are associated with higher critic scores even after controlling for runtime and kid-friendliness, while Action & Adventure is significantly negative and linked to lower critic ratings.

More linear regression models on newly engineered features

Model 4: Model 3’s features + days_to_stream

In [17]:
feat_model4 = feat_model3 + ["days_to_stream"]

lr4 = LinearRegression().fit(X_train[feat_model4], y_train)
y4_pred = lr4.predict(X_test[feat_model4])
score_regression(y_test, y4_pred)

R2      -0.017956
MAE     24.608636
RMSE    28.629507
dtype: float64

Model 4 starts from the best previous structure (runtime, kid_friendly, and all genre dummies) and adds days_to_stream, which captures how long a movie stayed in theaters before streaming. Longer theatrical windows often signal bigger or more “prestige” releases that critics might score differently, so it’s a reasonable feature to try for better prediction. This adds release-timing information while still keeping the full genre detail.

Model 5 – simplify genres using p-values

In [19]:
feat_model5 = feat_model2 + ["days_to_stream"] + ['Animation', 'Art House & International', 'Classics', 'Comedy', 'Documentary', 'Drama', 
             'Musical & Performing Arts', 'Special Interest', 'Sports & Fitness', 'Television', 'Western', 'Action & Adventure']
lr5 = LinearRegression().fit(X_train[feat_model5], y_train)
y5_pred = lr5.predict(X_test[feat_model5])
score_regression(y_test, y5_pred)

R2      -0.026366
MAE     24.701971
RMSE    28.747522
dtype: float64

Model 5 keeps the core variables (runtime, kid_friendly, days_to_stream) but drops genre dummies that had large p-values in the statsmodels output (e.g., Anime & Manga, Cult Movies, Faith & Spirituality, etc.). The idea is to remove genres that don’t appear strongly related to critic ratings, simplifying the model and potentially reducing noise/overfitting while keeping the more informative genres.

Model 6 – add other engineered features

In [20]:
feat_model6 = feat_model5 + ["kid_family", "holiday_blockbuster"]

lr6 = LinearRegression().fit(X_train[feat_model6], y_train)
y6_pred = lr6.predict(X_test[feat_model6])
score_regression(y_test, y6_pred)

R2      -0.024067
MAE     24.673948
RMSE    28.715312
dtype: float64

Model 6 builds on Model 5 by adding more targeted engineered features: kid_family (family-friendly and labeled “Kids & Family”) and holiday_blockbuster (holiday release + blockbuster genres). These combine existing columns into higher-level signals about very specific kinds of releases that critics may treat differently, so it’s a sensible final attempt to improve predictive power beyond the simpler linear effects.

Across the six models, Model 3 (runtime + kid_friendly + all genre dummies) performs the best: it’s the only model with a positive R^2 (~0.03) and it also has the lowest MAE (~24.0) and RMSE (~27.9). Adding extra features like days_to_stream or the more engineered indicators (kid_family, holiday_blockbuster) in Models 4–6 doesn’t improve test performance and slightly worsens the metrics, suggesting they mostly add noise or overfit. From the statsmodels output, runtime_in_minutes and kid_friendly both have highly significant coefficients, and several genre dummies have small p-values and sizable coefficients, so these runtime and genre effects appear to be the features that do the most consistent work in predicting critic_rating.